# 01 — Pygame CE : introduction

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- initialiser une fenêtre Pygame et gérer la boucle principale ;
- dessiner des formes (rect, circle, line, polygon) ;
- charger et afficher des images (blit, convert_alpha) ;
- gérer les événements clavier et souris ;
- détecter les collisions avec `Rect` ;
- utiliser `pygame.sprite.Sprite` et `Group` ;
- jouer des sons et de la musique ;
- animer avec le delta time.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- Python, classes, héritage ;
- les bases de la programmation orientée objet.

Ce que ce notebook **ne couvrira pas** :

- les moteurs physiques (Box2D, Pymunk) ;
- les jeux 3D (Panda3D, Ursina) ;
- les shaders et le rendu GPU.

## Plan

1. Installation
2. Fenêtre et boucle principale
3. Événements
4. Dessiner des formes
5. Charger des images
6. Rect et collisions
7. Sprites et Groups
8. Sons et musique
9. Delta time
10. Mini jeu squelette
11. Synthèse
12. Exercices
13. Ressources externes

### Exemple avancé : système de particules

Les systèmes de particules sont omniprésents dans les jeux vidéo :
explosions, fumée, étincelles, neige... Voici une implémentation avec Pygame.

In [ ]:
import pygame
import random
import math

class Particle(pygame.sprite.Sprite):
    """Particule avec durée de vie, vélocité et couleur."""
    def __init__(self, x, y, color=(255, 200, 50)):
        super().__init__()
        self.image = pygame.Surface((4, 4), pygame.SRCALPHA)
        self.image.fill(color)
        self.rect = self.image.get_rect(center=(x, y))
        angle = random.uniform(0, 2 * math.pi)
        speed = random.uniform(1, 5)
        self.vx = math.cos(angle) * speed
        self.vy = math.sin(angle) * speed
        self.lifetime = random.randint(20, 60)
        self.age = 0

    def update(self):
        self.age += 1
        if self.age >= self.lifetime:
            self.kill()
            return
        self.rect.x += self.vx
        self.rect.y += self.vy
        self.vy += 0.1  # gravité
        # Fondu
        alpha = int(255 * (1 - self.age / self.lifetime))
        self.image.set_alpha(max(0, alpha))

class ParticleEmitter:
    def __init__(self, group):
        self.group = group

    def emit(self, x, y, count=20, color=(255, 200, 50)):
        for _ in range(count):
            self.group.add(Particle(x, y, color))

# Utilisation dans une boucle de jeu :
# particles = pygame.sprite.Group()
# emitter = ParticleEmitter(particles)
# emitter.emit(400, 300)  # explosion au clic
# particles.update()
# particles.draw(screen)
print('Système de particules défini')

### Exercice : jeu de l'esquive

Créez un mini-jeu où :
1. Le joueur contrôle un carré bleu (flèches du clavier)
2. Des obstacles rouges tombent du haut de l'écran
3. Le score augmente avec le temps de survie
4. Collision = game over

Utilisez les `Sprite` et `Group` de Pygame pour gérer les collisions.

In [ ]:
import pygame
import random

class Player(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        self.image = pygame.Surface((30, 30))
        self.image.fill((50, 100, 255))
        self.rect = self.image.get_rect(center=(400, 550))
        self.speed = 5

    def update(self, keys):
        if keys[pygame.K_LEFT]: self.rect.x -= self.speed
        if keys[pygame.K_RIGHT]: self.rect.x += self.speed
        self.rect.clamp_ip(pygame.Rect(0, 0, 800, 600))

class Obstacle(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        w = random.randint(20, 60)
        self.image = pygame.Surface((w, 20))
        self.image.fill((255, 50, 50))
        self.rect = self.image.get_rect(
            center=(random.randint(0, 800), -20)
        )
        self.speed = random.uniform(2, 6)

    def update(self):
        self.rect.y += self.speed
        if self.rect.top > 600:
            self.kill()

def run_dodge_game():
    pygame.init()
    screen = pygame.display.set_mode((800, 600))
    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 36)

    player = Player()
    all_sprites = pygame.sprite.Group(player)
    obstacles = pygame.sprite.Group()
    score = 0
    spawn_timer = 0

    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        keys = pygame.key.get_pressed()
        player.update(keys)
        obstacles.update()

        spawn_timer += 1
        if spawn_timer > 15:
            obs = Obstacle()
            obstacles.add(obs)
            all_sprites.add(obs)
            spawn_timer = 0

        if pygame.sprite.spritecollide(player, obstacles, False):
            running = False  # Game Over

        score += 1
        screen.fill((20, 20, 40))
        all_sprites.draw(screen)
        screen.blit(font.render(f'Score: {score}', True, (255,255,255)), (10, 10))
        pygame.display.flip()
        clock.tick(60)

    pygame.quit()
    print(f'Game Over! Score: {score}')

# Décommenter pour jouer :
# run_dodge_game()
print('Jeu de l\'esquive défini')

### Exercice : animation de sprite sheet

Dans les vrais jeux, les animations utilisent des *sprite sheets* :
une image contenant toutes les frames côte à côte.

1. Créez une classe `AnimatedSprite` qui découpe une surface en frames
2. La méthode `update()` change de frame selon un timer
3. Testez avec une animation simple (carré qui change de couleur)

In [ ]:
import pygame

class AnimatedSprite(pygame.sprite.Sprite):
    """Sprite animé à partir de frames."""
    def __init__(self, frames, fps=10):
        super().__init__()
        self.frames = frames
        self.frame_index = 0
        self.image = self.frames[0]
        self.rect = self.image.get_rect()
        self.animation_speed = 1000 / fps  # ms par frame
        self.last_update = 0

    def update(self, current_time):
        if current_time - self.last_update >= self.animation_speed:
            self.frame_index = (self.frame_index + 1) % len(self.frames)
            self.image = self.frames[self.frame_index]
            self.last_update = current_time

# Création de frames simples (carrés colorés)
# colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0)]
# frames = []
# for color in colors:
#     surf = pygame.Surface((32, 32))
#     surf.fill(color)
#     frames.append(surf)
# sprite = AnimatedSprite(frames, fps=4)
print('AnimatedSprite défini')

### Bonnes pratiques Pygame

- **Limitez le framerate** : `clock.tick(60)` pour 60 FPS
- **Utilisez `convert()` et `convert_alpha()`** sur les images pour accélérer le rendu
- **Groupez les sprites** : `Group.update()` et `Group.draw()` sont optimisés
- **Séparez logique et rendu** : la logique ne doit pas dépendre du FPS
- **Utilisez `delta_time`** pour un mouvement fluide indépendant du framerate
- **Nettoyez** : appelez `pygame.quit()` à la fin

---

## 1. Installation

In [ ]:
# pip install pygame-ce
import pygame
print(f"Pygame CE {pygame.version.ver}")

---

## 2. Fenêtre et boucle principale

Tout jeu Pygame suit le même squelette : init, boucle, quit.

In [ ]:
# Squelette minimal (ne pas lancer dans un notebook)
SQUELETTE = '''
import pygame
pygame.init()
screen = pygame.display.set_mode((800, 600))
pygame.display.set_caption("Mon jeu")
clock = pygame.time.Clock()
running = True

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    screen.fill((0, 0, 0))  # fond noir
    # dessiner ici...
    pygame.display.flip()
    clock.tick(60)  # 60 FPS max

pygame.quit()
'''
print('Squelette défini (800x600, 60 FPS)')

### Points clés

- `pygame.init()` : initialise tous les sous-systèmes
- `display.set_mode(size)` : crée la fenêtre
- `display.flip()` : met à jour l’écran entier
- `clock.tick(60)` : limite le FPS et retourne le temps écoulé (ms)

---

## 3. Événements

| Type | Description |
|---|---|
| `QUIT` | Fermeture de la fenêtre |
| `KEYDOWN` / `KEYUP` | Touche enfoncée / relâchée |
| `MOUSEBUTTONDOWN` / `MOUSEBUTTONUP` | Clic souris |
| `MOUSEMOTION` | Déplacement souris |
| `pygame.USEREVENT` | Événement personnalisé |

In [ ]:
# Gestion des touches
# if event.type == pygame.KEYDOWN:
#     if event.key == pygame.K_LEFT:
#         print('Gauche')
#     elif event.key == pygame.K_SPACE:
#         print('Espace')
print('K_LEFT, K_RIGHT, K_UP, K_DOWN, K_SPACE, K_ESCAPE')

In [ ]:
# Polling continu (pour mouvement fluide)
# keys = pygame.key.get_pressed()
# if keys[pygame.K_LEFT]:
#     player.x -= speed * dt
print('key.get_pressed() pour le polling continu')

In [ ]:
# Souris
# pos = pygame.mouse.get_pos()  # (x, y)
# pressed = pygame.mouse.get_pressed()  # (left, middle, right)
print('mouse.get_pos(), mouse.get_pressed()')

---

## 4. Dessiner des formes

In [ ]:
# pygame.draw.rect(screen, (255, 0, 0), (x, y, w, h))
# pygame.draw.rect(screen, (255, 0, 0), (x, y, w, h), width=2)  # contour
print('draw.rect(surface, color, rect, width=0)')

In [ ]:
# pygame.draw.circle(screen, (0, 255, 0), (cx, cy), radius)
# pygame.draw.line(screen, (255, 255, 255), (x1, y1), (x2, y2), width)
# pygame.draw.polygon(screen, color, [(x1,y1), (x2,y2), (x3,y3)])
print('draw.circle, draw.line, draw.polygon')

In [ ]:
# Texte
# font = pygame.font.SysFont('Arial', 24)
# surface = font.render('Score: 42', True, (255, 255, 255))
# screen.blit(surface, (10, 10))
print('font.render() -> Surface, puis blit()')

---

## 5. Charger des images

In [ ]:
# image = pygame.image.load('player.png').convert_alpha()
# screen.blit(image, (x, y))
print('convert_alpha() pour transparence + performance')

### `convert()` vs `convert_alpha()`

- `convert()` : format optimal sans transparence
- `convert_alpha()` : format optimal avec canal alpha

In [ ]:
# Redimensionner
# image = pygame.transform.scale(image, (new_w, new_h))
# image = pygame.transform.rotate(image, angle_degrees)
print('transform.scale(), transform.rotate()')

---

## 6. Rect et collisions

`pygame.Rect` est l’outil central pour positionner et détecter les collisions.

In [ ]:
r1 = pygame.Rect(0, 0, 50, 50)
r2 = pygame.Rect(25, 25, 50, 50)

print(f"Collision rect-rect: {r1.colliderect(r2)}")
print(f"Point dans rect: {r1.collidepoint(10, 10)}")
print(f"Centre: {r1.center}")

In [ ]:
# Propriétés utiles
print(f"top={r1.top}, bottom={r1.bottom}, left={r1.left}, right={r1.right}")
print(f"topleft={r1.topleft}, center={r1.center}, size={r1.size}")

In [ ]:
# Clamp : rester dans la fenêtre
screen_rect = pygame.Rect(0, 0, 800, 600)
r1.clamp_ip(screen_rect)  # modifie r1 en place
print(f"Après clamp: {r1}")

---

## 7. Sprites et Groups

`Sprite` est la classe de base pour les entités du jeu.

In [ ]:
class Player(pygame.sprite.Sprite):
    def __init__(self) -> None:
        super().__init__()
        self.image = pygame.Surface((40, 40))
        self.image.fill((255, 255, 255))
        self.rect = self.image.get_rect(center=(400, 300))
        self.speed = 300  # pixels/seconde

    def update(self, dt: float) -> None:
        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]:
            self.rect.x -= self.speed * dt
        if keys[pygame.K_RIGHT]:
            self.rect.x += self.speed * dt

print('Player défini avec image, rect, update(dt)')

In [ ]:
# Group : gère update + draw pour tous les sprites
# all_sprites = pygame.sprite.Group(player, enemy1, enemy2)
# all_sprites.update(dt)
# all_sprites.draw(screen)
print('Group.update() et Group.draw() en une ligne')

In [ ]:
# Collisions entre groupes
# hits = pygame.sprite.spritecollide(player, enemies, dokill=True)
# for enemy in hits:
#     print('Collision !')
print('spritecollide(sprite, group, dokill)')

---

## 8. Sons et musique

In [ ]:
# pygame.mixer.init()
# son = pygame.mixer.Sound('beep.wav')
# son.play()  # effet sonore court
# son.set_volume(0.5)
print('mixer.Sound() pour les effets sonores')

In [ ]:
# Musique de fond
# pygame.mixer.music.load('music.ogg')
# pygame.mixer.music.play(loops=-1)  # -1 = boucle infinie
# pygame.mixer.music.set_volume(0.3)
# pygame.mixer.music.stop()
print('mixer.music pour la musique de fond')

---

## 9. Delta time

Pour des vitesses indépendantes du FPS, multiplier par `dt` (en secondes).

In [ ]:
# Dans la boucle :
# dt = clock.tick(60) / 1000.0  # secondes
# player.x += player.speed * dt  # 300 pixels/seconde quel que soit le FPS
print('dt = clock.tick(fps) / 1000 -> secondes')

### Pourquoi c’est important

- Sans `dt` : à 30 FPS le joueur va 2x moins vite qu’à 60 FPS.
- Avec `dt` : la vitesse est en pixels/seconde, constante.

---

## 10. Mini jeu squelette

Un carré qui se déplace au clavier et reste dans la fenêtre.

In [ ]:
MINI_JEU = '''
import pygame
pygame.init()
screen = pygame.display.set_mode((800, 600))
clock = pygame.time.Clock()
player = pygame.Rect(400, 300, 40, 40)
speed = 300  # pixels/seconde
running = True

while running:
    dt = clock.tick(60) / 1000
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT]:  player.x -= speed * dt
    if keys[pygame.K_RIGHT]: player.x += speed * dt
    if keys[pygame.K_UP]:    player.y -= speed * dt
    if keys[pygame.K_DOWN]:  player.y += speed * dt
    player.clamp_ip(screen.get_rect())
    screen.fill((30, 30, 50))
    pygame.draw.rect(screen, (200, 200, 255), player)
    pygame.display.flip()

pygame.quit()
'''
print('Mini jeu : carré mobile avec clamp')

---

## Synthèse

| Élément | Pygame CE | Notes |
|---|---|---|
| Init | `pygame.init()` + `display.set_mode(size)` | |
| Boucle | `for e in event.get(): ...` + `display.flip()` | |
| FPS | `clock.tick(60)` | Retourne dt en ms |
| Dessin | `draw.rect`, `draw.circle`, `draw.line` | |
| Image | `image.load(path).convert_alpha()` | + `blit()` |
| Texte | `font.render(text, True, color)` | + `blit()` |
| Collision | `Rect.colliderect()`, `collidepoint()` | |
| Sprite | `class X(Sprite)` + `Group` | `update()`, `draw()` |
| Son | `mixer.Sound(path).play()` | |
| Musique | `mixer.music.load()/play()` | |

### Règles à retenir

- **Toujours `flip()`** après avoir dessiné.
- **`convert()` / `convert_alpha()`** pour les performances.
- **`clock.tick(60)`** limite le FPS et donne le dt.
- **Multiplier par `dt`** pour des vitesses en pixels/seconde.
- **`Sprite` + `Group`** pour structurer les entités.

---

---

## Timers et evenements personnalises

In [ ]:
# Creer un evenement periodique
# SPAWN_ENEMY = pygame.USEREVENT + 1
# pygame.time.set_timer(SPAWN_ENEMY, 2000)  # toutes les 2 secondes
#
# Dans la boucle :
# if event.type == SPAWN_ENEMY:
#     spawn_enemy()
print('USEREVENT + set_timer(event_id, milliseconds)')

Les timers sont utiles pour spawner des ennemis, des power-ups ou declencher des animations.

---

## Surfaces et transparence

In [ ]:
# Surface avec transparence (per-pixel alpha)
# surf = pygame.Surface((100, 100), pygame.SRCALPHA)
# surf.fill((255, 0, 0, 128))  # rouge semi-transparent
# screen.blit(surf, (50, 50))
print('SRCALPHA pour la transparence per-pixel')

In [ ]:
# Colorkey (transparence d'une couleur)
# surf = pygame.Surface((100, 100))
# surf.fill((255, 0, 255))  # fond magenta
# surf.set_colorkey((255, 0, 255))  # magenta = transparent
print('set_colorkey() pour rendre une couleur transparente')

---

## Groupes avances

In [ ]:
# GroupSingle : contient au max 1 sprite
# player_group = pygame.sprite.GroupSingle(player)
#
# spritecollideany : booleen rapide
# if pygame.sprite.spritecollideany(player, walls):
#     print('Collision !')
print('GroupSingle, spritecollideany, groupcollide')

In [ ]:
# Collision entre deux groupes
# hits = pygame.sprite.groupcollide(bullets, enemies, True, True)
# Le 1er True = supprimer les bullets en collision
# Le 2e True = supprimer les enemies en collision
print('groupcollide(g1, g2, dokill1, dokill2)')

---

## Texte et polices avancees

In [ ]:
# Police par defaut
# font = pygame.font.Font(None, 36)
# Police systeme
# font = pygame.font.SysFont('arial', 24, bold=True)
# Police fichier
# font = pygame.font.Font('myfont.ttf', 28)
# 
# text_surface = font.render('Score: 42', True, (255, 255, 255))
# text_rect = text_surface.get_rect(center=(400, 30))
# screen.blit(text_surface, text_rect)
print('Font(None, size), SysFont(name, size), Font(path, size)')

## Exercices

### Exercice 1 *(facile)* — Fenêtre noire

Ouvrez une fenêtre 640x480 noire qui se ferme quand on clique sur la croix.

In [ ]:
import pygame

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import pygame
pygame.init()
screen = pygame.display.set_mode((640, 480))
running = True
while running:
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False
    screen.fill((0, 0, 0))
    pygame.display.flip()
pygame.quit()
```

</details>

### Exercice 2 *(facile)* — Carré au centre

Affichez un carré rouge 50x50 au centre d’une fenêtre 640x480.

In [ ]:
import pygame

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import pygame
pygame.init()
screen = pygame.display.set_mode((640, 480))
rect = pygame.Rect(0, 0, 50, 50)
rect.center = (320, 240)
running = True
while running:
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False
    screen.fill((0, 0, 0))
    pygame.draw.rect(screen, (255, 0, 0), rect)
    pygame.display.flip()
pygame.quit()
```

</details>

### Exercice 3 *(moyen)* — Mouvement au clavier

Le carré se déplace avec les flèches du clavier à 300 px/s. Utilisez le delta time.

In [ ]:
import pygame

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import pygame
pygame.init()
screen = pygame.display.set_mode((640, 480))
clock = pygame.time.Clock()
rect = pygame.Rect(0, 0, 50, 50)
rect.center = (320, 240)
speed = 300
running = True
while running:
    dt = clock.tick(60) / 1000
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT]:  rect.x -= speed * dt
    if keys[pygame.K_RIGHT]: rect.x += speed * dt
    if keys[pygame.K_UP]:    rect.y -= speed * dt
    if keys[pygame.K_DOWN]:  rect.y += speed * dt
    rect.clamp_ip(screen.get_rect())
    screen.fill((0, 0, 0))
    pygame.draw.rect(screen, (255, 0, 0), rect)
    pygame.display.flip()
pygame.quit()
```

</details>

### Exercice 4 *(moyen)* — Balle rebondissante

Une balle (cercle) qui rebondit sur les bords de la fenêtre.

In [ ]:
import pygame

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import pygame
pygame.init()
screen = pygame.display.set_mode((640, 480))
clock = pygame.time.Clock()
x, y = 320.0, 240.0
vx, vy = 200.0, 150.0
radius = 15
running = True
while running:
    dt = clock.tick(60) / 1000
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False
    x += vx * dt
    y += vy * dt
    if x - radius < 0 or x + radius > 640:
        vx = -vx
    if y - radius < 0 or y + radius > 480:
        vy = -vy
    screen.fill((0, 0, 0))
    pygame.draw.circle(screen, (0, 255, 0), (int(x), int(y)), radius)
    pygame.display.flip()
pygame.quit()
```

</details>

### Exercice 5 *(difficile)* — Snake minimal

Implémentez un Snake ultra-simplifié :
- Tête qui se déplace sur une grille
- Mange une pomme rouge
- Grandit d’un segment

In [ ]:
import pygame

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import pygame
import random

pygame.init()
size = 20
screen = pygame.display.set_mode((400, 400))
clock = pygame.time.Clock()
snake = [(200, 200)]
dx, dy = size, 0
apple = (random.randrange(0, 400, size), random.randrange(0, 400, size))
running = True
while running:
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False
        if e.type == pygame.KEYDOWN:
            if e.key == pygame.K_LEFT and dx == 0:  dx, dy = -size, 0
            if e.key == pygame.K_RIGHT and dx == 0: dx, dy = size, 0
            if e.key == pygame.K_UP and dy == 0:    dx, dy = 0, -size
            if e.key == pygame.K_DOWN and dy == 0:  dx, dy = 0, size
    head = (snake[-1][0] + dx, snake[-1][1] + dy)
    snake.append(head)
    if head == apple:
        apple = (random.randrange(0, 400, size), random.randrange(0, 400, size))
    else:
        snake.pop(0)
    screen.fill((0, 0, 0))
    for s in snake:
        pygame.draw.rect(screen, (0, 255, 0), (*s, size, size))
    pygame.draw.rect(screen, (255, 0, 0), (*apple, size, size))
    pygame.display.flip()
    clock.tick(10)
pygame.quit()
```

</details>

### Approfondir : gestion des états de jeu

Un vrai jeu a plusieurs "écrans" : menu, jeu, pause, game over.
Le pattern State Machine permet de les gérer proprement.

In [ ]:
import pygame

class GameState:
    """Classe de base pour un état de jeu."""
    def handle_events(self, events): pass
    def update(self, dt): pass
    def draw(self, screen): pass

class MenuState(GameState):
    def __init__(self, game):
        self.game = game
        self.font = pygame.font.Font(None, 48)

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.KEYDOWN and event.key == pygame.K_RETURN:
                self.game.state = PlayState(self.game)

    def draw(self, screen):
        screen.fill((20, 20, 60))
        text = self.font.render('Appuyez sur ENTREE', True, (255, 255, 255))
        screen.blit(text, (200, 280))

class PlayState(GameState):
    def __init__(self, game):
        self.game = game
        self.x = 400

    def handle_events(self, events):
        for event in events:
            if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                self.game.state = MenuState(self.game)

    def update(self, dt):
        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]: self.x -= 300 * dt
        if keys[pygame.K_RIGHT]: self.x += 300 * dt

    def draw(self, screen):
        screen.fill((30, 80, 30))
        pygame.draw.rect(screen, (255, 200, 50), (self.x-15, 280, 30, 30))

class Game:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((800, 600))
        self.clock = pygame.time.Clock()
        self.state = MenuState(self)
        self.running = True

    def run(self):
        while self.running:
            dt = self.clock.tick(60) / 1000
            events = pygame.event.get()
            for e in events:
                if e.type == pygame.QUIT:
                    self.running = False
            self.state.handle_events(events)
            self.state.update(dt)
            self.state.draw(self.screen)
            pygame.display.flip()
        pygame.quit()

# Game().run()
print('State Machine de jeu définie')

### Exercice : texte et score

Créez un mini-jeu de clic :
1. Des cibles (cercles) apparaissent à des positions aléatoires
2. Un clic sur une cible augmente le score et la fait disparaître
3. Chaque cible a un timer (2 secondes) avant de disparaître
4. Affichage du score, du temps restant, et du meilleur score

In [ ]:
import pygame
import random
import time

class ClickTarget:
    def __init__(self):
        self.x = random.randint(50, 750)
        self.y = random.randint(50, 550)
        self.radius = random.randint(15, 35)
        self.created = time.time()
        self.lifetime = 2.0
        self.color = (random.randint(100,255), random.randint(100,255), 50)

    def is_expired(self):
        return time.time() - self.created > self.lifetime

    def contains(self, mx, my):
        return (mx - self.x)**2 + (my - self.y)**2 <= self.radius**2

    def draw(self, screen):
        remaining = 1 - (time.time() - self.created) / self.lifetime
        alpha = int(255 * max(0, remaining))
        pygame.draw.circle(screen, self.color, (self.x, self.y), self.radius)
        pygame.draw.circle(screen, (255,255,255), (self.x, self.y), self.radius, 2)

def click_game():
    pygame.init()
    screen = pygame.display.set_mode((800, 600))
    font = pygame.font.Font(None, 36)
    clock = pygame.time.Clock()
    targets = []
    score = 0
    spawn_timer = 0

    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT: running = False
            if event.type == pygame.MOUSEBUTTONDOWN:
                mx, my = event.pos
                for t in targets[:]:
                    if t.contains(mx, my):
                        targets.remove(t)
                        score += 10
                        break

        spawn_timer += 1
        if spawn_timer > 30:
            targets.append(ClickTarget())
            spawn_timer = 0
        targets = [t for t in targets if not t.is_expired()]

        screen.fill((20, 20, 40))
        for t in targets: t.draw(screen)
        screen.blit(font.render(f'Score: {score}', True, (255,255,255)), (10, 10))
        pygame.display.flip()
        clock.tick(60)
    pygame.quit()

# click_game()
print('Click game défini')

### Exercice bonus : tilemap

Les tilemaps permettent de construire des niveaux à partir d'une grille de tuiles.
Voici un loader minimaliste.

In [ ]:
import pygame

class TileMap:
    """Tilemap basique depuis une grille 2D."""
    TILE_SIZE = 32
    COLORS = {
        0: None,           # vide
        1: (100, 60, 30),  # terre
        2: (50, 150, 50),  # herbe
        3: (100, 100, 200),# eau
    }

    def __init__(self, grid):
        self.grid = grid
        self.height = len(grid)
        self.width = len(grid[0]) if grid else 0

    def draw(self, screen, offset_x=0, offset_y=0):
        for row in range(self.height):
            for col in range(self.width):
                tile_id = self.grid[row][col]
                color = self.COLORS.get(tile_id)
                if color:
                    rect = pygame.Rect(
                        col * self.TILE_SIZE + offset_x,
                        row * self.TILE_SIZE + offset_y,
                        self.TILE_SIZE, self.TILE_SIZE
                    )
                    pygame.draw.rect(screen, color, rect)
                    pygame.draw.rect(screen, (0,0,0), rect, 1)  # bordure

    def get_solid_rects(self):
        """Retourne les rectangles des tuiles solides (pour collisions)."""
        rects = []
        for row in range(self.height):
            for col in range(self.width):
                if self.grid[row][col] in (1, 2):  # solides
                    rects.append(pygame.Rect(
                        col * self.TILE_SIZE, row * self.TILE_SIZE,
                        self.TILE_SIZE, self.TILE_SIZE
                    ))
        return rects

# Exemple de niveau
level = [
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,0,2,2,0,0,0,0,0],
    [0,0,0,0,0,0,0,2,2,0],
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,2,2,0,0,0,0,0,0],
    [0,0,0,0,0,3,3,0,0,0],
    [2,2,2,2,2,1,1,2,2,2],
    [1,1,1,1,1,1,1,1,1,1],
]
tilemap = TileMap(level)
print(f'Map: {tilemap.width}x{tilemap.height} tiles')
print(f'Tiles solides: {len(tilemap.get_solid_rects())}')

### Astuce : structure d'un projet Pygame

Pour un projet de taille moyenne, organisez votre code :
```
mon_jeu/
  main.py          # point d'entrée
  settings.py      # constantes (SCREEN_SIZE, FPS, COLORS)
  game.py          # classe Game (boucle principale)
  sprites/
    player.py      # classe Player(Sprite)
    enemies.py     # classes d'ennemis
    particles.py   # système de particules
  levels/
    loader.py      # chargement de tilemaps
    level_01.json  # données du niveau
  assets/
    images/        # sprites, backgrounds
    sounds/        # effets sonores, musique
    fonts/         # polices
```

### Exercice : ajout de son

Pygame gère le son via `pygame.mixer`.

1. Initialisez le mixer
2. Créez des sons via `pygame.mixer.Sound`
3. Jouez un son lors d'une action (tir, collecte, collision)

In [ ]:
import pygame

# Initialisation du son
# pygame.mixer.init(frequency=44100, size=-16, channels=2)

# Charger un son
# sound_collect = pygame.mixer.Sound('coin.wav')
# sound_collect.set_volume(0.5)

# Musique de fond
# pygame.mixer.music.load('background.mp3')
# pygame.mixer.music.set_volume(0.3)
# pygame.mixer.music.play(-1)  # -1 = boucle infinie

# Jouer un effet sonore
# sound_collect.play()

# Générer un son programmatiquement (bip)
import numpy as np
# sample_rate = 44100
# duration = 0.1  # secondes
# t = np.linspace(0, duration, int(sample_rate * duration))
# wave = (np.sin(2 * np.pi * 440 * t) * 32767).astype(np.int16)
# stereo = np.column_stack((wave, wave))
# beep = pygame.sndarray.make_sound(stereo)
# beep.play()

print('API audio Pygame documentée')

### Ressources Pygame-CE

- **Documentation** : [pyga.me/docs](https://pyga.me/docs)
- **Tutoriels** : pygame.org/wiki
- **Assets gratuits** : kenney.nl, opengameart.org
- **Exemples inclus** : `python -m pygame.examples.aliens`
- **Distribution** : PyInstaller, cx_Freeze, Briefcase

---

## Ressources externes

- [Pygame CE — documentation](https://pyga.me/)
- [Pygame tutorials](https://www.pygame.org/wiki/tutorials)
- [Pygame CE GitHub](https://github.com/pygame-community/pygame-ce)